TBA

### import packages

In [3]:
import ast
import pandas as pd
import os
import numpy as np

### Load Data

In [9]:
#set szenario
szenario_year = '2021'
szenario_name = 'run_' + szenario_year #+ '_with_RU'

In [10]:
#output specifications
output_file_path = os.path.join('..', '..', '01_data', '02_output_data', '02_unidirectional_results', '02_paper_ESR', '02_prepared_results')
outout_file_name = '\flow_analysis_ESR_2026_' + szenario_name + '.xlsx'
#create full ouput paths
output_file_path_excel  = output_file_path + outout_file_name
full_output_file_path = os.path.abspath(os.path.join(os.getcwd(), output_file_path_excel))

In [11]:
# Specify the path to the results Excel file
input_results_file_path_results = os.path.join('..', '..','01_data', '02_output_data', '02_unidirectional_results', '02_paper_ESR', '01_raw_results')
results_file_name = '\outputs_ESR_2026_' + szenario_name + '.xlsx'


# Specify the path to the input data Excel file
input_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed', '02_paper_ESR')
input_file_name = '\inputs_ESR_2026_' + szenario_name + '.xlsx'

# Specify the path to the raw edges data Excel file
file_name_edges_cap_cost_raw = '\inputs_ESR_2026_edges_raw.xlsx'

#create full input paths
#results of the optimization
input_results_file_path_excel  = input_results_file_path_results + results_file_name
full_input_results_path = os.path.abspath(os.path.join(os.getcwd(), input_results_file_path_excel))

#inputs of the optimization
input_file_path_excel  = input_file_path + input_file_name
full_input_file_path = os.path.abspath(os.path.join(os.getcwd(), input_file_path_excel))

#edges raw path
file_path_edges_cap_costs_raw  = input_file_path + file_name_edges_cap_cost_raw
full_file_path_edges_cap_costs_raw = os.path.abspath(os.path.join(os.getcwd(), file_path_edges_cap_costs_raw))

In [12]:
results_methan_flow_raw_df = pd.read_excel(full_input_results_path, sheet_name='flows_methane_edges')

### code

In [13]:
def filter_flows_between_regions(
    df: pd.DataFrame,
    from_regions,
    to_regions,
    edge_col: str = "Edge",
    flow_col: str = "Flow",
    commodity: str | None = None,
    drop_zero_flows: bool = False,
):
    """
    Return all rows where Edge goes from any region in `from_regions`
    to any region in `to_regions`.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    from_regions : str or list-like
        Origin region(s), e.g. "UA" or ["UA", "MD"].
    to_regions : str or list-like
        Destination region(s), e.g. ["CZ", "BE", "DE"].
    edge_col : str
        Column containing edges, e.g. ('UA', 'SK') or "('UA', 'SK')".
    flow_col : str
        Flow column name.
    commodity : str or None
        Optional filter, e.g. "Methane".
    drop_zero_flows : bool
        If True, remove rows where Flow == 0.

    Returns
    -------
    pd.DataFrame
        Filtered dataframe with extra columns: origin, destination.
    """
    # Normalize inputs
    if isinstance(from_regions, str):
        from_regions = [from_regions]
    if isinstance(to_regions, str):
        to_regions = [to_regions]

    from_regions = set(from_regions)
    to_regions = set(to_regions)

    out = df.copy()

    # Parse edge values safely whether they are tuples or strings
    def parse_edge(x):
        if isinstance(x, tuple) and len(x) == 2:
            return x
        if isinstance(x, str):
            return ast.literal_eval(x)
        raise ValueError(f"Unsupported edge format: {x!r}")

    out[["origin", "destination"]] = out[edge_col].apply(
        lambda x: pd.Series(parse_edge(x))
    )

    mask = out["origin"].isin(from_regions) & out["destination"].isin(to_regions)

    if commodity is not None:
        mask &= out["Commodity"].eq(commodity)

    if drop_zero_flows:
        mask &= out[flow_col].ne(0)

    return out.loc[mask].reset_index(drop=True)



def _parse_edge(x):
    """Parse Edge values that are either tuples or strings like "('DE', 'CH')"."""
    if isinstance(x, tuple) and len(x) == 2:
        return x
    if isinstance(x, str):
        return ast.literal_eval(x)
    raise ValueError(f"Unsupported edge format: {x!r}")


def edges_above_share(
    df: pd.DataFrame,
    regions,
    min_share: float,
    edge_col: str = "Edge",
    share_col: str = "Share",
    commodity: str | None = None,
    connection_mode: str = "both",
    drop_zero_flows: bool = True,
):
    """
    Filter edges by region connectivity and keep only rows with Share >= min_share.

    connection_mode:
        'both'   -> both origin and destination in regions
        'import' -> origin outside, destination inside
        'export' -> origin inside, destination outside
        'either' -> at least one side inside
    """
    regions = set(regions)
    out = df.copy()

    out[["origin", "destination"]] = out[edge_col].apply(
        lambda x: pd.Series(_parse_edge(x))
    )

    origin_in = out["origin"].isin(regions)
    destination_in = out["destination"].isin(regions)

    if connection_mode == "both":
        mask = origin_in & destination_in
    elif connection_mode == "import":
        mask = (~origin_in) & destination_in
    elif connection_mode == "export":
        mask = origin_in & (~destination_in)
    elif connection_mode == "either":
        mask = origin_in | destination_in
    else:
        raise ValueError("connection_mode must be: 'both', 'import', 'export', or 'either'")

    if commodity is not None:
        mask &= out["Commodity"].eq(commodity)

    if drop_zero_flows:
        mask &= out["Flow"].ne(0)

    out = out.loc[mask].copy()

    if share_col not in out.columns:
        raise KeyError(f"'{share_col}' not found in dataframe.")

    out = out.loc[out[share_col] >= min_share].sort_values(
        by=share_col, ascending=False
    )

    return out.reset_index(drop=True)

def lng_import_supplier_shares(
    df: pd.DataFrame,
    lng_import_nodes,
    edge_col: str = "Edge",
    flow_col: str = "Flow",
    share_col: str = "Share",
    commodity: str | None = None,
    min_edge_share: float | None = None,
    drop_zero_flows: bool = True,
):
    """
    Return LNG import edges plus supplier shares into each LNG import node.

    Output columns include:
        origin, destination, Flow, Share, supplier_flow, node_total_flow, supplier_share

    supplier_share = supplier_flow / total_flow_into_that_LNG_node
    """
    lng_import_nodes = set(lng_import_nodes)
    out = df.copy()

    out[["origin", "destination"]] = out[edge_col].apply(
        lambda x: pd.Series(_parse_edge(x))
    )

    mask = out["destination"].isin(lng_import_nodes)

    if commodity is not None:
        mask &= out["Commodity"].eq(commodity)

    if drop_zero_flows:
        mask &= out[flow_col].ne(0)

    if min_edge_share is not None:
        if share_col not in out.columns:
            raise KeyError(f"'{share_col}' not found in dataframe.")
        mask &= out[share_col].ge(min_edge_share)

    out = out.loc[mask].copy()

    # Aggregate supplier flow by (LNG node, origin)
    supplier_totals = (
        out.groupby(["destination", "origin"], as_index=False)[flow_col]
        .sum()
        .rename(columns={flow_col: "supplier_flow"})
    )

    # Total inflow into each LNG node
    node_totals = (
        supplier_totals.groupby("destination", as_index=False)["supplier_flow"]
        .sum()
        .rename(columns={"supplier_flow": "node_total_flow"})
    )

    supplier_totals["supplier_share"] = (
        supplier_totals["supplier_flow"]
        / supplier_totals.groupby("destination")["supplier_flow"].transform("sum")
    )

    # Merge supplier shares back onto the edge-level rows
    out = out.merge(
        supplier_totals,
        on=["destination", "origin"],
        how="left",
    ).merge(
        node_totals,
        on="destination",
        how="left",
    )

    out = out.sort_values(
        ["destination", "supplier_share", flow_col],
        ascending=[True, False, False],
    )

    return out.reset_index(drop=True)

### execution

In [18]:
#regional groups
europe = [
    "CZ",
    "BE",
    "DE",
    "FR",
    "NL",
    "AT",
    "SI",
    "HR",
    "PL",
    "SK",
    "IT",
    "ES",
    "RO",
    "HU",
    "RS",
    "BA",
    "MD",
    "UA",
    "BG",
    "CH",
    "UK",
    "LV",
    "TR",
    "FI",
]

northern_africa = [
    "DZ",
    "TN",
    "LY",
    "MA",
]

lng_import_nodes = [
    "EL_LNG_imp",
    "IT_LNG_imp",
    "BE_LNG_imp",
    "TR_LNG_imp",
    "ES_LNG_imp",
    "HR_LNG_imp",
    "PL_LNG_imp",
    "UK_LNG_imp",
    "LT_LNG_imp",
    "NL_LNG_imp",
    "FR_LNG_imp",
    "PT_LNG_imp",
]

lng_import_countries = [
    "EL",
    "IT",
    "BE",
    "TR",
    "ES",
    "HR",
    "PL",
    "UK",
    "LT",
    "NL",
    "FR",
    "PT",
]

In [19]:
#Russia to Europe
ru_to_europe = filter_flows_between_regions(
    results_methan_flow_raw_df,
    from_regions="RU",
    to_regions=europe,
    commodity="Methane"
)

#Northern Africa to Europe
NA_to_europe = filter_flows_between_regions(
    results_methan_flow_raw_df,
    from_regions=northern_africa,
    to_regions=europe,
    commodity="Methane"
)

#Norway to Europe
no_to_europe = filter_flows_between_regions(
    results_methan_flow_raw_df,
    from_regions="NO",
    to_regions=europe,
    commodity="Methane"
)

#Caspean Region to Europe
cr_to_europe = filter_flows_between_regions(
    results_methan_flow_raw_df,
    from_regions="CR",
    to_regions=europe,
    commodity="Methane"
)

In [20]:
ru_to_europe

,Commodity,Edge,Flow,Share,origin,destination
0,Methane,"('RU', 'DE')",0.000000,0.000000,RU,DE
1,Methane,"('RU', 'UA')",81410.643395,0.167199,RU,UA
2,Methane,"('RU', 'TR')",0.000000,0.000000,RU,TR
3,Methane,"('RU', 'LV')",0.000000,0.000000,RU,LV
4,Methane,"('RU', 'FI')",18910.099333,0.235493,RU,FI


In [21]:
total_RU_to_europe = ru_to_europe["Flow"].sum()
total_RU_to_europe

np.float64(100320.74272777312)

In [30]:
RU_real = 167*9.77*1000
reduction = (1-(total_RU_to_europe/RU_real))*100
round(reduction, 2)

np.float64(93.85)

In [78]:
NA_to_europe

,Commodity,Edge,Flow,Share,origin,destination
0,Methane,"('MA', 'ES')",64624.344207,0.399849,MA,ES
1,Methane,"('DZ', 'ES')",123041.500000,1.000000,DZ,ES
2,Methane,"('LY', 'IT')",74716.270809,0.747163,LY,IT
3,Methane,"('TN', 'IT')",180000.000000,1.000000,TN,IT


In [79]:
total_NA_to_europe = NA_to_europe["Flow"].sum()
total_NA_to_europe

np.float64(442382.1150156051)

In [80]:
no_to_europe

,Commodity,Edge,Flow,Share,origin,destination
0,Methane,"('NO', 'NL')",206781.071033,0.587924,NO,NL
1,Methane,"('NO', 'DE')",127969.452970,0.184114,NO,DE
2,Methane,"('NO', 'FR')",208050.000000,1.000000,NO,FR
3,Methane,"('NO', 'BE')",178120.000000,1.000000,NO,BE
4,Methane,"('NO', 'UK')",397158.931507,0.725840,NO,UK


In [81]:
total_no_to_europe = no_to_europe["Flow"].sum()
total_no_to_europe

np.float64(1118079.455509577)

In [82]:
cr_to_europe

,Commodity,Edge,Flow,Share,origin,destination
0,Methane,"('CR', 'TR')",100000.0,1.0,CR,TR


In [83]:
total_cr_to_europe = cr_to_europe["Flow"].sum()
total_cr_to_europe

np.float64(99999.99999999994)

In [87]:
lng_to_europe = filter_flows_between_regions(
    results_methan_flow_raw_df,
    from_regions=lng_import_nodes,
    to_regions=lng_import_countries,
    commodity="Methane",
    drop_zero_flows=True
)

In [88]:
lng_to_europe

,Commodity,Edge,Flow,Share,origin,destination
0,Methane,"('PT_LNG_imp', 'PT')",59939.164912,0.840414,PT_LNG_imp,PT
1,Methane,"('TR_LNG_imp', 'TR')",261991.640650,0.615044,TR_LNG_imp,TR
2,Methane,"('EL_LNG_imp', 'EL')",13418.101000,0.196200,EL_LNG_imp,EL
3,Methane,"('FR_LNG_imp', 'FR')",47841.380669,0.148387,FR_LNG_imp,FR
4,Methane,"('ES_LNG_imp', 'ES')",162505.359118,0.276757,ES_LNG_imp,ES
5,Methane,"('IT_LNG_imp', 'IT')",154366.000000,1.000000,IT_LNG_imp,IT


In [89]:
total_lng_to_europe = lng_to_europe["Flow"].sum()
total_lng_to_europe

np.float64(700061.6463490791)

In [64]:
europe_pipeline_use = edges_above_share(
    results_methan_flow_raw_df,
    regions=europe,
    min_share=0.8,
    commodity="Methane",
    connection_mode="either"
)

In [65]:
europe_pipeline_use

,Commodity,Edge,Flow,Share,origin,destination
0,Methane,"('DZ', 'ES')",123041.50000,1.000000,DZ,ES
1,Methane,"('NO', 'FR')",208050.00000,1.000000,NO,FR
2,Methane,"('LY', 'IT')",60000.00000,1.000000,LY,IT
3,Methane,"('RU', 'DE')",635830.00000,1.000000,RU,DE
4,Methane,"('HR', 'SI')",2810.50000,1.000000,HR,SI
5,Methane,"('TN', 'IT')",120000.00000,1.000000,TN,IT
6,Methane,"('BG', 'EL')",42723.25000,1.000000,BG,EL
7,Methane,"('RU', 'TR')",57056.80000,1.000000,RU,TR
8,Methane,"('NO', 'BE')",178120.00000,1.000000,NO,BE
9,Methane,"('CR', 'TR')",73260.00000,1.000000,CR,TR


In [66]:
lng_shares = lng_import_supplier_shares(
    results_methan_flow_raw_df,
    lng_import_nodes=lng_import_nodes,
    commodity="Methane",
    min_edge_share=0.0
)

In [67]:
lng_shares

,Commodity,Edge,Flow,Share,origin,destination,supplier_flow,supplier_share,node_total_flow
0,Methane,"('AF_LNG_exp', 'EL_LNG_imp')",13418.101000,0.000013,AF_LNG_exp,EL_LNG_imp,13418.101000,1.000000,13418.101000
1,Methane,"('AF_LNG_exp', 'ES_LNG_imp')",113102.804945,0.000113,AF_LNG_exp,ES_LNG_imp,113102.804945,1.000000,113102.804945
2,Methane,"('USA_LNG_exp', 'FR_LNG_imp')",122557.651478,0.000123,USA_LNG_exp,FR_LNG_imp,122557.651478,1.000000,122557.651478
3,Methane,"('AF_LNG_exp', 'IT_LNG_imp')",154366.000000,0.000154,AF_LNG_exp,IT_LNG_imp,154366.000000,1.000000,154366.000000
4,Methane,"('USA_LNG_exp', 'PT_LNG_imp')",59939.164912,0.000060,USA_LNG_exp,PT_LNG_imp,59939.164912,1.000000,59939.164912
5,Methane,"('TT_LNG_exp', 'TR_LNG_imp')",92277.017281,0.000092,TT_LNG_exp,TR_LNG_imp,92277.017281,0.319594,288731.640650
6,Methane,"('AF_LNG_exp', 'TR_LNG_imp')",78976.158780,0.000079,AF_LNG_exp,TR_LNG_imp,78976.158780,0.273528,288731.640650
7,Methane,"('USA_LNG_exp', 'TR_LNG_imp')",59834.776958,0.000060,USA_LNG_exp,TR_LNG_imp,59834.776958,0.207233,288731.640650
8,Methane,"('EG_LNG_exp', 'TR_LNG_imp')",57643.687631,0.000058,EG_LNG_exp,TR_LNG_imp,57643.687631,0.199645,288731.640650


In [68]:
fr_lng = lng_shares[lng_shares["destination"] == "FR_LNG_imp"]

In [69]:
fr_lng

,Commodity,Edge,Flow,Share,origin,destination,supplier_flow,supplier_share,node_total_flow
2,Methane,"('USA_LNG_exp', 'FR_LNG_imp')",122557.651478,0.000123,USA_LNG_exp,FR_LNG_imp,122557.651478,1.0,122557.651478
